# 联动标识检查 —— 备件不联动清单 vs EPS量产件最新输机价格对比

**说明：** 本 notebook 分步执行，每一步独立为一个 cell，方便逐节调试和修改。

**数据源：**
- `备件_不联动清单_整改及跟踪项.xlsx`：备件不联动清单
- `eps3_mass.xlsx`：EPS量产件输机价格数据


## 步骤1：读取《备件_不联动清单_整改及跟踪项.xlsx》，筛选"对策"为"不联动→联动（推进中）"，构造 key1

key1 = 量产件号 + "_" + 供应商编码


In [ ]:
import pandas as pd

# ---------- 读取备件不联动清单 ----------
df1 = pd.read_excel('备件_不联动清单_整改及跟踪项.xlsx')
print(f'File1 总行数: {len(df1)}')
print(f'File1 列名: {list(df1.columns)}')
print(f'对策 唯一值: {df1["对策"].unique().tolist()}')

# 筛选：对策 == "不联动→联动（推进中）"
target_duice = '不联动→联动（推进中）'
df1_filtered = df1[df1['对策'] == target_duice].copy()
print(f'\n筛选后行数 ({target_duice}): {len(df1_filtered)}')

# 构造 key1：量产件号 + "_" + 供应商编码
# 统一转为字符串，去除前后空格，确保匹配一致性
df1_filtered['key1'] = (
    df1_filtered['量产件号'].astype(str).str.strip()
    + '_'
    + df1_filtered['供应商编码'].astype(str).str.strip()
)
print(f'key1 去重数量: {df1_filtered["key1"].nunique()}')

# 预览前几行
df1_filtered[['量产件号', '供应商编码', '备件裸价', '对策', 'key1']].head(10)


## 步骤2：读取《eps3_mass.xlsx》，筛选"状态"为"已审批"，构造 key2

key2 = 零件号 + "_" + 供应商编码


In [ ]:
# ---------- 读取 eps3_mass ----------
df2 = pd.read_excel('eps3_mass.xlsx')
print(f'File2 总行数: {len(df2)}')
print(f'File2 列名: {list(df2.columns)}')
print(f'状态 唯一值: {df2["状态"].unique().tolist()}')

# 筛选：状态 == "已审批"
target_status = '已审批'
df2_filtered = df2[df2['状态'] == target_status].copy()
print(f'\n筛选后行数 (状态="{target_status}"): {len(df2_filtered)}')

# 构造 key2：零件号 + "_" + 供应商编码
df2_filtered['key2'] = (
    df2_filtered['零件号'].astype(str).str.strip()
    + '_'
    + df2_filtered['供应商编码'].astype(str).str.strip()
)
print(f'key2 去重数量: {df2_filtered["key2"].nunique()}')

# 预览前几行
df2_filtered[['零件号', '供应商编码', '零件裸价', '状态', '生效日期', '失效日期', '创建时间', 'key2']].head(10)


## 步骤3：按 key2 分组，保留"失效日期"最晚的记录

**规则：**
1. 如果"生效日期"与"失效日期"不在同一年，先将"失效日期"调整至"生效日期"当年的最后一天（12月31日）
2. 按 **失效日期（调整后）降序**、再按 **创建时间降序** 排序
3. 每组取第一条（即失效日期最晚，相同时取创建时间最晚）


In [ ]:
# ---------- 日期列转换 ----------
df2_filtered['生效日期'] = pd.to_datetime(df2_filtered['生效日期'], errors='coerce')
df2_filtered['失效日期'] = pd.to_datetime(df2_filtered['失效日期'], errors='coerce')
df2_filtered['创建时间'] = pd.to_datetime(df2_filtered['创建时间'], errors='coerce')

# ---------- 调整失效日期 ----------
# 如果"生效日期"与"失效日期"不在同一年，将"失效日期"调整至"生效日期"当年的最后一天
def adjust_expiry(row):
    eff = row['生效日期']
    exp = row['失效日期']
    if pd.notna(eff) and pd.notna(exp) and eff.year != exp.year:
        # 调整至生效日期当年的 12月31日 23:59:59
        return pd.Timestamp(year=eff.year, month=12, day=31, hour=23, minute=59, second=59)
    return exp

df2_filtered['失效日期_调整后'] = df2_filtered.apply(adjust_expiry, axis=1)

# 查看调整情况（可选）
adjusted_count = (df2_filtered['失效日期'] != df2_filtered['失效日期_调整后']).sum()
print(f'失效日期被调整的记录数: {adjusted_count}')
if adjusted_count > 0:
    print('\n调整示例（前5条被调整的记录）:')
    mask = df2_filtered['失效日期'] != df2_filtered['失效日期_调整后']
    display(df2_filtered.loc[mask, ['key2', '生效日期', '失效日期', '失效日期_调整后']].head(5))

# ---------- 按 key2 分组，取失效日期最晚 + 创建时间最晚 ----------
df2_latest = (
    df2_filtered
    .sort_values(['key2', '失效日期_调整后', '创建时间'], ascending=[True, False, False])
    .groupby('key2', as_index=False)
    .first()
)

print(f'\n分组去重后记录数: {len(df2_latest)}')

# 预览
cols_show = ['零件号', '供应商编码', '零件裸价', '生效日期', '失效日期', '失效日期_调整后', '创建时间', 'key2']
df2_latest[cols_show].head(10)


## 步骤4：匹配 key1 与 key2，比较"备件裸价"与"零件裸价"

- 以 key1 为基准，左连接 key2 的最新记录
- 新增列 `备件裸价与最新输机量产件对比`：值为"大于"、"等于"、"小于"


In [ ]:
# ---------- 准备 key2 的合并字段 ----------
# 只取匹配需要的列
df2_merge = df2_latest[['key2', '零件裸价', '价格通知书号', '生效日期', '失效日期', '失效日期_调整后']].copy()
df2_merge = df2_merge.rename(columns={'零件裸价': '最新量产零件裸价'})

# ---------- 左连接：key1 匹配 key2 ----------
df_result = df1_filtered.merge(
    df2_merge,
    how='left',
    left_on='key1',
    right_on='key2',
    suffixes=('', '_量产')
)

# ---------- 价格对比 ----------
def compare_price(row):
    spare_price = row['备件裸价']
    mass_price = row['最新量产零件裸价']
    if pd.isna(mass_price):
        return '无匹配'
    if spare_price > mass_price:
        return '大于'
    elif spare_price == mass_price:
        return '等于'
    else:
        return '小于'

df_result['备件裸价与最新输机量产件对比'] = df_result.apply(compare_price, axis=1)

# ---------- 汇总统计 ----------
print('===== 匹配与对比结果汇总 =====')
print(f'key1 总记录数: {len(df_result)}')
print(f'成功匹配到量产件记录: {df_result["最新量产零件裸价"].notna().sum()}')
print(f'未匹配到量产件记录: {df_result["最新量产零件裸价"].isna().sum()}')
print()
print('价格对比分布:')
print(df_result['备件裸价与最新输机量产件对比'].value_counts().to_string())

# ---------- 预览结果 ----------
cols_result = [
    '量产件号', '供应商编码', '备件裸价', '最新量产零件裸价',
    '备件裸价与最新输机量产件对比', '价格通知书号',
    '生效日期', '失效日期', '失效日期_调整后', '对策'
]
df_result[cols_result].head(20)


## （可选）导出结果到 Excel


In [ ]:
from datetime import datetime
today = datetime.now().strftime('%Y%m%d')
output_file = f'./联动标识检查结果/160项持续跟踪检查结果_{today}.xlsx'
df_result.to_excel(output_file, index=False)
print(f'结果已导出到: {output_file}')
